# Step 05: Model Export, Evaluation & Comparison

This notebook exports the fine-tuned PyTorch Lightning checkpoint (`.ckpt`) to **ONNX format (`.onnx`)**, runs benchmark synthesis, and compares performance metrics (RTF, speed, quality) side-by-side against the baseline model.

### Pipeline Steps:
1. Export best `.ckpt` to `.onnx` and JSON config
2. Synthesize benchmark test sentences with fine-tuned model
3. Compare baseline vs fine-tuned metrics (`metrics/experiment_comparison.csv`)
4. Listen to side-by-side audio samples in Colab

In [ ]:
# 1. Export Best Checkpoint to ONNX
import os
from pathlib import Path

ckpt_dir = Path('/content/drive/MyDrive/Arabic-Piper/checkpoints/experiment001')
best_ckpts = sorted(list(ckpt_dir.glob('*.ckpt')))

if best_ckpts:
    target_ckpt = str(best_ckpts[-1])
    output_onnx = '/content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx'
    print(f"Exporting checkpoint: {target_ckpt}")
    !python scripts/export_model.py --checkpoint "{target_ckpt}" --output-onnx "{output_onnx}"
else:
    print("No checkpoint found in checkpoints directory.")

In [ ]:
# 2. Benchmark Fine-Tuned Model
!python scripts/benchmark.py \
    --model /content/drive/MyDrive/Arabic-Piper/outputs/experiment001/ar_JO_finetuned.onnx \
    --sentences benchmark/benchmark_sentences.txt \
    --output-dir /content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark

In [ ]:
# 3. Compare Baseline vs Fine-Tuned Metrics
!python scripts/evaluate.py \
    --baseline-report /content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_report.json \
    --finetuned-report /content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark/benchmark_report.json \
    --output-csv /content/drive/MyDrive/Arabic-Piper/metrics/experiment001_comparison.csv

In [ ]:
# 4. Side-by-Side Audio Comparison
from IPython.display import Audio, display, HTML
from pathlib import Path

base_audio = Path('/content/drive/MyDrive/Arabic-Piper/outputs/baseline_benchmark/benchmark_01.wav')
ft_audio = Path('/content/drive/MyDrive/Arabic-Piper/outputs/finetuned_benchmark/benchmark_01.wav')

if base_audio.exists() and ft_audio.exists():
    display(HTML("<h3>Baseline Model Audio:</h3>"))
    display(Audio(str(base_audio)))
    display(HTML("<h3>Fine-Tuned Model Audio:</h3>"))
    display(Audio(str(ft_audio)))